In [1]:
#import libraries
from google.colab import files
import pandas as pd
import io
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [5]:
#Read the dataset from drive

uploaded = files.upload()

df = pd.read_csv(io.BytesIO(uploaded[list(uploaded.keys())[0]]))

#Display first 5 rows
df.head()

Saving Churn_Modelling.csv to Churn_Modelling.csv


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
# Finding Missing Values

print(df.isnull().sum())

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [7]:
#Handling Missing values

#Fill numerical missing values with mean
df.fillna(df.mean(numeric_only=True), inplace=True)

#Fill categorical missing values with mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values handled")

Missing values handled


/tmp/ipykernel_9756/3503678590.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [8]:
#Check for Duplicates

print("Duplicate Rows:", df.duplicated().sum())

#Remove duplicate rows
df = df.drop_duplicates()

print("Duplicates removed")

Duplicate Rows: 0
Duplicates removed


In [9]:
#Detect Outliers

#Using IQR Method
Q1 = df.select_dtypes(include=np.number).quantile(0.25)
Q3 = df.select_dtypes(include=np.number).quantile(0.75)
IQR = Q3 - Q1

outliers = ((df.select_dtypes(include=np.number) < (Q1 - 1.5 * IQR)) |
            (df.select_dtypes(include=np.number) > (Q3 + 1.5 * IQR)))

print("Outliers detected:")
print(outliers.sum())

Outliers detected:
RowNumber             0
CustomerId            0
CreditScore          15
Age                 359
Tenure                0
Balance               0
NumOfProducts        60
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Exited             2037
dtype: int64


In [10]:
#Normalize the dataset

#Selecting numerical columns
num_cols = df.select_dtypes(include=np.number).columns

#Min-Max Normalization
scaler = MinMaxScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])

print("Dataset normalized")

Dataset normalized


In [11]:
#split the dataset into input and output

#Assuming last column is target/output
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Input Features:")
print(X.head())

print("Output Variable:")
print(y.head())

Input Features:
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender       Age  \
0     0.0000    0.275616  Hargrave        0.538    France  Female  0.324324   
1     0.0001    0.326454      Hill        0.516     Spain  Female  0.310811   
2     0.0002    0.214421      Onio        0.304    France  Female  0.324324   
3     0.0003    0.542636      Boni        0.698    France  Female  0.283784   
4     0.0004    0.688778  Mitchell        1.000     Spain  Female  0.337838   

   Tenure   Balance  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  
0     0.2  0.000000       0.000000        1.0             1.0         0.506735  
1     0.1  0.334031       0.000000        0.0             1.0         0.562709  
2     0.8  0.636357       0.666667        1.0             0.0         0.569654  
3     0.1  0.000000       0.333333        0.0             0.0         0.469120  
4     0.2  0.500246       0.000000        1.0             1.0         0.395400  
Output Variable:
0    1

In [12]:
#splitting the data for training & Testing

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training and Testing data created")

Training and Testing data created


In [13]:
#Print the training data and testing data

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

print("\nTraining Data:")
print(X_train.head())

print("\nTesting Data:")
print(X_test.head())

X_train Shape: (8000, 13)
X_test Shape: (2000, 13)
y_train Shape: (8000,)
y_test Shape: (2000,)

Training Data:
      RowNumber  CustomerId   Surname  CreditScore Geography  Gender  \
9254   0.925493    0.141666      P'an        0.672    France    Male   
1561   0.156116    0.802727      Leak        0.564   Germany    Male   
1670   0.167017    0.605199     Green        0.418     Spain    Male   
6087   0.608761    0.660261  Chukwudi        0.422    France  Female   
6669   0.666967    0.928837  Chinomso        0.334    France    Male   

           Age  Tenure   Balance  NumOfProducts  HasCrCard  IsActiveMember  \
9254  0.189189     0.6  0.000000       0.333333        1.0             1.0   
1561  0.324324     0.4  0.476786       0.333333        1.0             1.0   
1670  0.081081     0.3  0.457317       0.000000        1.0             0.0   
6087  0.121622     0.9  0.540606       0.000000        1.0             0.0   
6669  0.513514     0.9  0.566554       0.000000        0.0       